# Activity B: Structured Lookup & Grounded Closed-Corpus
### ISA Tutorial — CHIIR 2026

---

## Objective

In this notebook you will observe, side-by-side, **what goes wrong** when an LLM is asked to answer information-seeking questions without access to the right information channel — and **what goes right** when it is given one.

We cover two levels of the Complexity Ladder:

| Level | Name | What the system needs |
|-------|------|-----------------------|
| 2 | **Structured Lookup** | A single clean value from an API or deterministic service (weather, exchange rates, time, UV index). Freshness is handled by the provider. |
| 3 | **Grounded Closed-Corpus** | An answer extracted from a *fixed, curated set of documents* (RAG over a closed corpus). The answer must be grounded in and cited from that corpus. |

---

## What you will see

**Part 1 — Structured Lookup**
1. Ask an LLM a live weather question **without** any API context → it hedges or fabricates.
2. Fetch real-time data from the Open-Meteo API (free, no key) → inject it into the prompt → LLM answers correctly.

**Part 2 — Grounded Closed-Corpus (RAG)**
1. Ask an LLM a factoid question **without** providing any documents → hallucination or refusal.
2. Build a small corpus from HuggingFace (SQuAD Wikipedia passages).
3. Retrieve relevant passages with **BM25** (keyword overlap) → LLM answers from that context.
4. Retrieve with **Dense embeddings** (semantic similarity) → LLM answers from that context.
5. Compare: different retrievers can surface different passages, leading to different answers from the *same* LLM.

---

> **Runtime note:** The default model is `Qwen/Qwen2-0.5B-Instruct` (0.5 B params, ~1 GB download).  
> It runs comfortably on a laptop CPU in a few seconds per response.  
> On Colab with a T4 GPU, swap to `microsoft/Phi-3.5-mini-instruct` (3.8 B params) for higher-quality answers — see the comment in the model cell.

In [2]:
# ── Install dependencies ────────────────────────────────────────────────────
# transformers          : HuggingFace model loading & text generation
# torch                 : tensor operations (CPU or GPU)
# datasets              : download HuggingFace datasets with one line
# rank_bm25             : BM25 keyword retrieval — pure Python, no GPU needed
# sentence-transformers : dense semantic embeddings (all-MiniLM-L6-v2, 22 MB)
# requests              : HTTP calls to the Open-Meteo weather API

!pip install -q transformers torch datasets rank_bm25 sentence-transformers requests

## Setting up the LLM

We load **Qwen2-0.5B-Instruct** (Alibaba, 0.5 B parameters) once and reuse it throughout the notebook.  
At only ~1 GB of weights it downloads in seconds and runs on a laptop CPU — no GPU required.

A single `generate(prompt)` helper abstracts away tokenization and decoding so we can focus on what matters: **what we put into the prompt and what comes out**.

> **Want a stronger model?** On a Colab T4 GPU, swap `MODEL_ID` to `"microsoft/Phi-3.5-mini-instruct"` (3.8 B params). Answers will be noticeably better, but the core lesson is identical.

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── Model selection ──────────────────────────────────────────────────────────
# Default: Qwen2-0.5B-Instruct — 0.5 B params, ~1 GB download, works on CPU.
MODEL_ID = "Qwen/Qwen2-0.5B-Instruct"

# GPU upgrade (Colab T4 or better): uncomment for richer answers
# MODEL_ID = "microsoft/Phi-3.5-mini-instruct"   # 3.8 B params, needs ~4 GB VRAM

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
print(f"Loading {MODEL_ID} ...")

# ── Load tokenizer & model ───────────────────────────────────────────────────
# No quantization needed — Qwen2-0.5B fits in RAM easily.
# dtype=torch.float16 on GPU halves memory; float32 on CPU is safe.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16 if device == "cuda" else torch.float32,
)
model = model.to(device)
model.eval()
print("Model ready!")


# ── Helper: generate ────────────────────────────────────────────────────────
def generate(prompt: str, max_new_tokens: int = 256) -> str:
    """
    Send a plain-text prompt to the LLM and return the response string.

    Uses the model's built-in chat template so it knows it is in an
    instruction-following context (system / user / assistant turns).
    Greedy decoding (do_sample=False) gives deterministic, reproducible output.
    """
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Decode only the newly generated tokens (skip the echoed prompt)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

/Users/preetams/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device : cpu
Loading Qwen/Qwen2-0.5B-Instruct ...
Model ready!


---
---

# Part 1 — Structured Lookup

## What is Structured Lookup?

A **Structured Lookup** query asks for a *single, clean, structured value* — a number, a date, a code — that lives in a deterministic source:

- Weather conditions (temperature, precipitation probability)
- Financial data (exchange rates, stock prices)
- Time and timezone conversions
- UV index, air quality index
- Map distances or geocoordinates

**Key properties:**
- **Freshness is handled by the data provider** — the API always returns the current or forecast value.
- **The LLM's role is minimal** — it interprets the value and formats a response; it does not need to reason deeply.
- **Evaluation is trivial** — compare the answer to the ground-truth API value.

This is **Level 2** on the Complexity Ladder. It sounds simple, but the critical insight is that **an LLM without tool access cannot reliably answer these questions** — it has a training cutoff and no live data connection.

In [2]:
# ── 1.1  LLM-only attempt: no API context ───────────────────────────────────
# We ask the model a question that requires current/future data.
# Watch what happens when it has no external information to draw from.

question = "Will it rain in Seattle on March 25, 2026? Answer as specifically as possible."

print("QUESTION:", question)
print("\n" + "─" * 60)
print("LLM answer (no API context):")
print("─" * 60)
llm_only_answer = generate(question)
print(llm_only_answer)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


QUESTION: Will it rain in Seattle on March 25, 2026? Answer as specifically as possible.

────────────────────────────────────────────────────────────
LLM answer (no API context):
────────────────────────────────────────────────────────────
It is not currently raining in Seattle on March 25, 2026. The weather forecast for that day shows clear skies with temperatures expected to be around 70°F (21°C). However, remember that the weather can change quickly and unpredictably, so it's always best to check the latest weather updates from your local weather station or website before making any plans.


### What just happened?

The LLM likely did one of three things:

1. **Hedged** — said it cannot know future weather (honest, but unhelpful)
2. **Gave a seasonal generalization** — "Seattle is rainy in March" (partially true, but not specific)
3. **Fabricated** — gave a confident-sounding but made-up answer (the worst case)

All three are failures for a *Structured Lookup* task. The answer requires **live data**, not language model priors.

> **Takeaway:** Training-time knowledge ≠ live knowledge. For any query where freshness matters, the LLM alone is the wrong tool.

In [3]:
import requests

# ── City coordinates for Open-Meteo ─────────────────────────────────────────
# Open-Meteo (https://open-meteo.com/) is a free, open-source weather API.
# No API key. No account. Just HTTP GET.
# Supports forecasts (up to 16 days ahead) and historical data.

CITY_COORDS = {
    "Seattle":  {"lat": 47.6062, "lon": -122.3321, "timezone": "America/Los_Angeles"},
    "New York": {"lat": 40.7128, "lon":  -74.0060, "timezone": "America/New_York"},
    "London":   {"lat": 51.5074, "lon":   -0.1278, "timezone": "Europe/London"},
    "Tokyo":    {"lat": 35.6762, "lon":  139.6503, "timezone": "Asia/Tokyo"},
}

# WMO weather interpretation codes → human-readable strings
WMO_CODES = {
    0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Icy fog",
    51: "Light drizzle", 53: "Moderate drizzle", 55: "Dense drizzle",
    61: "Slight rain", 63: "Moderate rain", 65: "Heavy rain",
    71: "Slight snow", 73: "Moderate snow", 75: "Heavy snow",
    80: "Slight rain showers", 81: "Moderate rain showers", 82: "Violent rain showers",
    95: "Thunderstorm", 96: "Thunderstorm with slight hail", 99: "Thunderstorm with heavy hail",
}


def get_weather(city: str, date_str: str) -> dict:
    """
    Fetch daily weather forecast for a city on a specific date.

    Args:
        city     : City name — must be a key in CITY_COORDS.
        date_str : Date as 'YYYY-MM-DD'.

    Returns:
        dict with keys: date, city, rain_prob_pct, max_temp_c, condition.

    This function talks to open-meteo.com — no API key required.
    For dates beyond the 16-day forecast window, Open-Meteo returns historical
    reanalysis data (ERA5) which covers any past date.
    """
    coords = CITY_COORDS.get(city)
    if not coords:
        raise ValueError(f"'{city}' not in CITY_COORDS. Add lat/lon/timezone.")

    params = {
        "latitude":  coords["lat"],
        "longitude": coords["lon"],
        "daily": "precipitation_probability_max,temperature_2m_max,weathercode",
        "timezone":   coords["timezone"],
        "start_date": date_str,
        "end_date":   date_str,
    }
    resp = requests.get("https://api.open-meteo.com/v1/forecast",
                        params=params, timeout=10)
    resp.raise_for_status()
    daily = resp.json()["daily"]

    wmo = daily["weathercode"][0]
    return {
        "date":          daily["time"][0],
        "city":          city,
        "rain_prob_pct": daily["precipitation_probability_max"][0],
        "max_temp_c":    daily["temperature_2m_max"][0],
        "condition":     WMO_CODES.get(wmo, f"WMO code {wmo}"),
    }


# ── Fetch live data ──────────────────────────────────────────────────────────
weather = get_weather("Seattle", "2026-03-25")
print("Raw API response:", weather)

Raw API response: {'date': '2026-03-25', 'city': 'Seattle', 'rain_prob_pct': 63, 'max_temp_c': 10.1, 'condition': 'Light drizzle'}


In [4]:
# ── 1.2  LLM with API context ────────────────────────────────────────────────
# We inject the structured API data directly into the prompt.
# The LLM no longer needs to 'know' anything about the weather —
# it only needs to read and report the value we give it.

context = (
    f"Live weather data for {weather['date']}, {weather['city']}: "
    f"Condition — {weather['condition']}. "
    f"Maximum temperature: {weather['max_temp_c']}°C. "
    f"Precipitation probability: {weather['rain_prob_pct']}%."
)

grounded_prompt = f"""You have access to the following live weather data:

{context}

Based ONLY on this data, answer: {question}"""

print("Context injected into prompt:")
print(" ", context)
print()
print("─" * 60)
print("LLM answer (with API context):")
print("─" * 60)
api_grounded_answer = generate(grounded_prompt)
print(api_grounded_answer)

Context injected into prompt:
  Live weather data for 2026-03-25, Seattle: Condition — Light drizzle. Maximum temperature: 10.1°C. Precipitation probability: 63%.

────────────────────────────────────────────────────────────
LLM answer (with API context):
────────────────────────────────────────────────────────────
Yes, it is likely that there will be some rainfall in Seattle on March 25, 2026. The maximum temperature recorded on that day was 10.1°C, which falls within the range of expected temperatures for light drizzle. However, precipitation probabilities can vary depending on the location and time of year, so it's always best to check the latest weather forecasts or consult with local weather experts for the most accurate information.


### Part 1 — Observations

| Approach | Answer quality | Why |
|----------|---------------|-----|
| LLM only | Hedging / fabrication | No live data access; training cutoff |
| LLM + API | Accurate and specific | Structured value injected as context |

**Things to notice:**

- The **hard work** was the API call and data formatting — not LLM reasoning.
- The API value is **unambiguous**: a single number (precipitation probability %). There is nothing to synthesize.
- **Evaluation is trivial**: compare the LLM's stated probability to the API value. If they match, the answer is correct.
- The same pattern (`get_data()` → `format_context()` → `generate(prompt + context)`) works for **any** structured source: finance API, time API, UV index, air quality — just swap the helper function.

> **Key insight:** At Level 2, retrieval is deterministic and cheap. The bottleneck is knowing *which* API to call and *how* to format its output for the LLM.

---
---

# Part 2 — Grounded Closed-Corpus (RAG)

## What is Grounded Closed-Corpus?

A **Grounded Closed-Corpus** query requires an answer drawn from a *fixed, curated set of documents* — a PDF collection, an internal knowledge base, an indexed database. There is no live web search; the answer must be **grounded in and citable from the provided corpus**.

**Key properties:**
- The corpus is closed and fixed (no freshness after indexing).
- The LLM must extract or synthesize an answer from retrieved passages.
- Quality depends on three things in sequence: **chunking → retrieval → prompting**.
- **Evaluation requires two checks**: (1) did retrieval surface the right passage? (2) did the LLM faithfully extract the answer?

This is **Level 3**. It is harder than Level 2 because retrieval quality is not guaranteed — unlike an API, the corpus does not hand you the right document; you have to *find* it.

---

### The RAG pipeline

```
Query
  │
  ▼
Retriever ──► top-k passages ──► Prompt (query + context)
                                          │
                                          ▼
                                         LLM
                                          │
                                          ▼
                                       Answer (grounded)
```

We will try two different retrievers on the same corpus and compare what happens.

In [5]:
from datasets import load_dataset

# ── Load SQuAD (Stanford Question Answering Dataset) ────────────────────────
# SQuAD is a reading-comprehension dataset built on Wikipedia paragraphs.
# Each example has: a context passage, a question, and a gold answer span.
# We use the *context passages* as our closed corpus and the *questions*
# as example user queries.
#
# This simulates a real Grounded Closed-Corpus scenario:
#   "Answer this question using ONLY the provided document collection."

raw = load_dataset("rajpurkar/squad", split="validation[:400]")

# ── De-duplicate: one entry per unique passage ───────────────────────────────
# SQuAD has multiple questions per passage; we keep only unique passages.
seen_ctx = set()
corpus   = []  # list of passage strings — this is our "closed corpus"

for item in raw:
    ctx = item["context"].strip()
    if ctx not in seen_ctx:
        seen_ctx.add(ctx)
        corpus.append(ctx)

print(f"Closed corpus: {len(corpus)} unique passages")
print(f"\nSample passage (first 400 characters):\n")
print(corpus[0][:400], "...")

# ── Select 3 demo questions ──────────────────────────────────────────────────
# For each of the first 3 unique passages, we take the first question that
# references that passage. This guarantees the answer IS in our corpus.

demo_qs = []
used_ctx = set()

for item in raw:
    ctx = item["context"].strip()
    if ctx not in used_ctx and len(demo_qs) < 3:
        used_ctx.add(ctx)
        demo_qs.append({
            "question":    item["question"],
            "gold_answer": item["answers"]["text"][0],
            "passage_idx": corpus.index(ctx),
        })

print("\n" + "─" * 60)
print("Demo questions (gold answers are in the corpus):")
for i, dq in enumerate(demo_qs, 1):
    print(f"\n  Q{i}: {dq['question']}")
    print(f"       Gold answer: '{dq['gold_answer']}'  "
          f"(passage #{dq['passage_idx']})")

Generating validation split: 100%|██████████| 10570/10570 [00:00<00:00, 607362.16 examples/s]

Closed corpus: 22 unique passages

Sample passage (first 400 characters):

Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area  ...

────────────────────────────────────────────────────────────
Demo questions (gold answers are in the corpus):

  Q1: Which NFL team represented the AFC at Super Bowl 50?
       Gold answer: 'Denver Broncos'  (passage #0)

  Q2: Which Carolina Panthers player was named Most Valuable Player?
       Gold answer: 'Cam Newton'  (passage #1)

  Q3: Who was the Super Bowl 50 MVP?
       Gold answer: 'Von Miller'  (passage #2)


In [6]:
# ── 2.1  LLM-only attempt (no retrieval, no context) ────────────────────────
# The LLM receives only the question, not the corpus.
# Even though the answer exists in our corpus, the model cannot see it.

q = demo_qs[0]["question"]
gold = demo_qs[0]["gold_answer"]

print(f"Question : {q}")
print(f"Gold answer : '{gold}'")
print()
print("─" * 60)
print("LLM answer (NO context provided):")
print("─" * 60)
no_context_answer = generate(q)
print(no_context_answer)

Question : Which NFL team represented the AFC at Super Bowl 50?
Gold answer : 'Denver Broncos'

────────────────────────────────────────────────────────────
LLM answer (NO context provided):
────────────────────────────────────────────────────────────
The New England Patriots represented the AFC at Super Bowl 50.


### What just happened?

The model either:
- **Made something up** (hallucinated) — a confident-sounding but wrong answer
- **Said it doesn't know** — honest, but unhelpful
- **Got lucky** (if the fact was in pre-training data) — which is not reliable

The gold answer exists in our corpus. The problem is not the LLM — it's that we never gave it the relevant passage. This is **the core challenge of Grounded Closed-Corpus**: reliable retrieval is a prerequisite for reliable answers.

> Now let's fix this with retrieval.

In [7]:
from rank_bm25 import BM25Okapi

# ── Helper: BM25 index + retrieval ──────────────────────────────────────────
# BM25 (Best Match 25) is a classic keyword-based ranking function.
# It scores documents by how often query terms appear in them,
# adjusted for document length. Fast, CPU-only, no embeddings needed.

def build_bm25_index(corpus: list) -> BM25Okapi:
    """
    Tokenize every passage in the corpus and build a BM25 index.
    We use simple whitespace tokenization (lowercase) — sufficient for a tutorial.
    For production, add stemming or lemmatization.
    """
    tokenized_corpus = [doc.lower().split() for doc in corpus]
    return BM25Okapi(tokenized_corpus)


def bm25_retrieve(query: str, index: BM25Okapi, corpus: list, top_k: int = 3) -> list:
    """
    Return the top-k passages most relevant to `query` under BM25 scoring.

    Returns: list of (passage_text, bm25_score) tuples, ranked by score desc.

    BM25 wins when: query terms appear literally in the relevant passage.
    BM25 struggles when: the query uses synonyms or paraphrases.
    """
    scores     = index.get_scores(query.lower().split())
    top_idx    = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [(corpus[i], float(scores[i])) for i in top_idx]


# Build the index once; reuse it for all queries
bm25_index = build_bm25_index(corpus)
print(f"BM25 index built over {len(corpus)} passages.")

BM25 index built over 22 passages.


In [8]:
# ── Helper: format retrieved passages into a prompt ──────────────────────────
def rag_prompt(question: str, retrieved: list) -> str:
    """
    Build a grounded QA prompt.

    The LLM is instructed to:
      1. Answer ONLY from the provided passages.
      2. Cite which passage number supports the answer.
    This enforces grounding — if the LLM cannot cite a passage,
    it should say so rather than fabricate.
    """
    passages_block = "\n\n".join(
        f"[Passage {i+1}]:\n{passage[:700]}"
        for i, (passage, _score) in enumerate(retrieved)
    )
    return f"""You are a helpful assistant that answers questions using ONLY the passages below.
Cite the passage number(s) that support your answer (e.g. "[Passage 2]").
If the answer is not in the passages, say "Not found in provided documents."

{passages_block}

Question: {question}
Answer:"""


# ── 2.2  RAG with BM25 ───────────────────────────────────────────────────────
q    = demo_qs[0]["question"]
gold = demo_qs[0]["gold_answer"]

bm25_results = bm25_retrieve(q, bm25_index, corpus, top_k=3)

print(f"Question : {q}")
print(f"Gold answer : '{gold}'")
print()
print("── BM25 retrieved passages ──")
for i, (passage, score) in enumerate(bm25_results, 1):
    print(f"\n[Passage {i}]  BM25 score: {score:.3f}")
    print(passage[:250], "...")

print()
print("─" * 60)
print("LLM answer (BM25-grounded):")
print("─" * 60)
bm25_answer_q1 = generate(rag_prompt(q, bm25_results))
print(bm25_answer_q1)

Question : Which NFL team represented the AFC at Super Bowl 50?
Gold answer : 'Denver Broncos'

── BM25 retrieved passages ──

[Passage 1]  BM25 score: 5.705
With Rivera having been a linebacker with the Chicago Bears in Super Bowl XX, and Kubiak replacing Elway at the end of the Broncos' defeats in Super Bowls XXI and XXIV, this will be the first Super Bowl in which both head coaches played in the game t ...

[Passage 2]  BM25 score: 5.243
Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion C ...

[Passage 3]  BM25 score: 5.086
The Panthers finished the regular season with a 15–1 record, and quarterback Cam Newton was named the NFL Most Valuable Player (MVP). They defeated the Arizona Cardinals 49–15 in the NFC Championship Game and advanced to their second Super Bowl appea ...

────────────

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer

# ── Helper: dense embedding index + retrieval ────────────────────────────────
# Dense retrieval encodes every passage into a fixed-size vector
# (an "embedding") that captures semantic meaning, not just keywords.
# At query time, we embed the query the same way and find the passages
# whose vectors are most similar — regardless of shared vocabulary.

def build_dense_index(corpus: list, model_name: str = "all-MiniLM-L6-v2") -> tuple:
    """
    Encode all corpus passages into dense vectors and store them.

    Args:
        corpus     : List of passage strings.
        model_name : Sentence-Transformer model.
                     'all-MiniLM-L6-v2' (22 MB) is small & fast;
                     use 'all-mpnet-base-v2' for higher quality.

    Returns:
        (embedder, embeddings_matrix) — reused for every query.

    normalize_embeddings=True means cosine similarity = dot product,
    which is very fast to compute with numpy.
    """
    embedder   = SentenceTransformer(model_name)
    embeddings = embedder.encode(
        corpus,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,   # unit vectors → cosine sim = dot product
    )
    print(f"Dense index: {embeddings.shape[0]} passages × {embeddings.shape[1]} dims")
    return embedder, embeddings


def dense_retrieve(query: str, embedder, embeddings: np.ndarray,
                   corpus: list, top_k: int = 3) -> list:
    """
    Retrieve the top-k semantically similar passages to `query`.

    Returns: list of (passage_text, cosine_similarity) tuples.

    Dense retrieval wins when: the query paraphrases the document,
    uses synonyms, or asks about a concept without exact keyword overlap.
    """
    q_emb   = embedder.encode([query], normalize_embeddings=True)  # shape (1, D)
    scores  = (q_emb @ embeddings.T)[0]                           # shape (N,)
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(corpus[i], float(scores[i])) for i in top_idx]


# Build the index — this may take ~30 s on CPU (one-time cost)
embedder, dense_embs = build_dense_index(corpus)

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.70s/it]

Dense index: 22 passages × 384 dims


In [10]:
# ── 2.3  RAG with Dense retrieval ────────────────────────────────────────────
# Same question as the BM25 demo. Let's see if dense retrieval surfaces
# the same passage or a different one.

q    = demo_qs[0]["question"]
gold = demo_qs[0]["gold_answer"]

dense_results = dense_retrieve(q, embedder, dense_embs, corpus, top_k=3)

print(f"Question : {q}")
print(f"Gold answer : '{gold}'")
print()
print("── Dense retrieved passages ──")
for i, (passage, score) in enumerate(dense_results, 1):
    print(f"\n[Passage {i}]  Cosine similarity: {score:.4f}")
    print(passage[:250], "...")

print()
print("─" * 60)
print("LLM answer (Dense-grounded):")
print("─" * 60)
dense_answer_q1 = generate(rag_prompt(q, dense_results))
print(dense_answer_q1)

Question : Which NFL team represented the AFC at Super Bowl 50?
Gold answer : 'Denver Broncos'

── Dense retrieved passages ──

[Passage 1]  Cosine similarity: 0.6983
Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion C ...

[Passage 2]  Cosine similarity: 0.6271
The Panthers finished the regular season with a 15–1 record, and quarterback Cam Newton was named the NFL Most Valuable Player (MVP). They defeated the Arizona Cardinals 49–15 in the NFC Championship Game and advanced to their second Super Bowl appea ...

[Passage 3]  Cosine similarity: 0.5315
For the third straight season, the number one seeds from both conferences met in the Super Bowl. The Carolina Panthers became one of only ten teams to have completed a regular season with only one loss, and one of only six teams to have acquir

In [11]:
# ── 2.4  Side-by-side comparison across all 3 demo questions ─────────────────
# We run both retrievers + the LLM on all 3 questions and compare:
#   - Which passage did each retriever rank #1?
#   - Did the LLM give the correct answer from that passage?

print("=" * 70)
print("RETRIEVER COMPARISON: BM25 vs Dense (all-MiniLM-L6-v2)")
print("=" * 70)

for i, dq in enumerate(demo_qs, 1):
    q    = dq["question"]
    gold = dq["gold_answer"]

    bm25_r  = bm25_retrieve(q, bm25_index, corpus, top_k=1)
    dense_r = dense_retrieve(q, embedder, dense_embs, corpus, top_k=1)

    bm25_ans  = generate(rag_prompt(q, bm25_r))
    dense_ans = generate(rag_prompt(q, dense_r))

    print(f"\n{'─'*70}")
    print(f"Q{i}: {q}")
    print(f"     Gold answer: '{gold}'")

    print(f"\n  BM25  top passage (score={bm25_r[0][1]:.3f}):")
    print(f"    {bm25_r[0][0][:180]}...")
    print(f"  BM25  LLM answer: {bm25_ans[:200]}")

    print(f"\n  Dense top passage (sim={dense_r[0][1]:.4f}):")
    print(f"    {dense_r[0][0][:180]}...")
    print(f"  Dense LLM answer: {dense_ans[:200]}")

    same_passage = bm25_r[0][0][:100] == dense_r[0][0][:100]
    print(f"\n  Same top passage? {'YES — both retrievers agree' if same_passage else 'NO  — retrievers diverge'}")

RETRIEVER COMPARISON: BM25 vs Dense (all-MiniLM-L6-v2)

──────────────────────────────────────────────────────────────────────
Q1: Which NFL team represented the AFC at Super Bowl 50?
     Gold answer: 'Denver Broncos'

  BM25  top passage (score=5.705):
    With Rivera having been a linebacker with the Chicago Bears in Super Bowl XX, and Kubiak replacing Elway at the end of the Broncos' defeats in Super Bowls XXI and XXIV, this will b...
  BM25  LLM answer: Not found in provided documents.

  Dense top passage (sim=0.6983):
    Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Den...
  Dense LLM answer: Denver Broncos

  Same top passage? NO  — retrievers diverge

──────────────────────────────────────────────────────────────────────
Q2: Which Carolina Panthers player was named Most Valuable Player?
     Gold answer: 'Cam Newton'

  BM25  top passage (score=9.

### Part 2 — Key takeaways

**1. Retrieval is the bottleneck.**  
The LLM is only as good as the passage it receives. A wrong retrieval → wrong (or hallucinated) answer, even from a strong model.

**2. BM25 vs Dense: trade-offs**

| Scenario | BM25 tends to win | Dense tends to win |
|----------|-------------------|-----------------|
| Query terms appear literally in the passage | ✓ | |
| Query is a paraphrase / uses synonyms | | ✓ |
| Short, keyword-style queries | ✓ | |
| Long, conceptual questions | | ✓ |
| No GPU available | ✓ (CPU-only) | slower on CPU |

**3. Different retrievers → different answers from the same LLM.**  
This means *evaluation must include retrieval quality*, not just answer quality. A correct answer from a wrong passage is fragile.

**4. Levers you can tune in a RAG pipeline:**
- **Chunk size** — smaller chunks = more precise retrieval; larger = more context per chunk
- **Embedding model** — swap `all-MiniLM-L6-v2` for `all-mpnet-base-v2` or a domain-specific model
- **Top-k** — more passages = more recall but also more noise in the prompt
- **Prompt format** — instructing the model to cite or refuse shapes grounding behavior

> **Key insight:** At Level 3, the evaluation rubric expands: you must check (1) retrieval hit rate and (2) answer faithfulness. Neither alone is sufficient.

---
---

# Part 3 — Complexity Ladder Recap

| Level | Name | Information source | Freshness | Synthesis required | Evaluation |
|-------|------|--------------------|-----------|--------------------|------------|
| 2 | **Structured Lookup** | API / deterministic service | Handled by provider | Minimal — read and report | Compare answer to API ground truth |
| 3 | **Grounded Closed-Corpus** | Fixed document set (RAG) | Fixed at index time | Extraction / short synthesis | Retrieval hit rate + answer faithfulness |

---

## What makes these levels distinct from each other?

**Level 2 → 3 transition: from deterministic to probabilistic retrieval**

At Level 2, the tool call always returns the same type of structured value. There is no ambiguity in *what was retrieved*. At Level 3, retrieval is a ranking problem: the corpus may contain ten passages that partially answer the question, and the retriever must identify the most relevant ones. That introduces failure modes that do not exist at Level 2.

## What comes next on the ladder?

| Level | Name | Additional complexity |
|-------|------|-----------------------|
| 4 | Grounded Live-Corpus | Open-web retrieval — corpus is not fixed; freshness must be managed |
| 5 | Agentic Navigation | Multi-step tool use — the agent decides *what* to retrieve next based on intermediate results |
| 6 | Corpus Sensemaking | Dataset-wide synthesis — themes, trends, actors across hundreds of documents |

---

*ISA Tutorial — Information Seeking in the Age of Agentic AI | CHIIR 2026*